# CBOND_ON Pipeline Outputs Check

This notebook inspects outputs of each pipeline stage: raw -> clean -> panel -> labels -> factors -> results.


In [14]:
from pathlib import Path
import json5
import pandas as pd

ROOT = Path(r'C:/Users/BaiYang/CBOND_ON')
paths_cfg = json5.loads((ROOT/'cbond_on/config/paths_config.json5').read_text(encoding='utf-8'))

# unified date controls
DATE_STR = '2025-01-03'
DATE = pd.to_datetime(DATE_STR).date()
DATE_STR, DATE


('2025-01-03', datetime.date(2025, 1, 3))

## 1) Raw Data (list a few)

In [15]:
raw_root = Path(paths_cfg['raw_data_root'])
sorted(raw_root.iterdir())[:5] if raw_root.exists() else f'Missing: {raw_root}'


[WindowsPath('D:/cbond_on/raw_data/market_cbond__daily_base'),
 WindowsPath('D:/cbond_on/raw_data/market_cbond__daily_deriv'),
 WindowsPath('D:/cbond_on/raw_data/market_cbond__daily_price'),
 WindowsPath('D:/cbond_on/raw_data/market_cbond__daily_rating'),
 WindowsPath('D:/cbond_on/raw_data/market_cbond__daily_twap')]

## 2) Cleaned Data (list a few)

In [16]:
clean_root = Path(paths_cfg['cleaned_data_root'])
sorted(clean_root.iterdir())[:5] if clean_root.exists() else f'Missing: {clean_root}'


[WindowsPath('D:/cbond_on/clean_data/LOBDS'),
 WindowsPath('D:/cbond_on/clean_data/snapshot')]

## 3) Panel Data (print head)

In [17]:
panel_root = Path(paths_cfg['panel_data_root'])
panel_files = list(panel_root.rglob(f"*{DATE.strftime('%Y%m%d')}*.parquet")) if panel_root.exists() else []
print('panel files:', len(panel_files))
if panel_files:
    df = pd.read_parquet(panel_files[0])
    display(df.head())


panel files: 1


trade_time  pre_close     last  \
dt                  code      seq                                           
2025-01-03 14:30:00 110052.SH 0   2025-01-02 14:13:03    132.368  129.629   
                              1   2025-01-02 14:13:06    132.368  129.651   
                              2   2025-01-02 14:13:09    132.368  129.651   
                              3   2025-01-02 14:13:12    132.368  129.560   
                              4   2025-01-02 14:13:15    132.368  129.500   

                                     open    high      low  close    volume  \
dt                  code      seq                                             
2025-01-03 14:30:00 110052.SH 0    134.44  134.95  129.561    0.0  538420.0   
                              1    134.44  134.95  129.561    0.0  538910.0   
                              2    134.44  134.95  129.561    0.0  538910.0   
                              3    134.44  134.95  129.560    0.0  539010.0   
                              4    134.44  134.95  129.500    0.0  539190.0   

                                        amount  num_trades  ...  ask_price4  \
dt                  code      seq                           ...               
2025-01-03 14:30:00 110052.SH 0    71056230.29       10601  ...     129.767   
                              1    71119742.52       10612  ...     129.649   
                              2    71119742.52       10612  ...     129.649   
                              3    71132700.57       10615  ...     129.607   
                              4    71156010.67       10620  ...     129.602   

                                   ask_volume4  bid_price4  bid_volume4  \
dt                  code      seq                                         
2025-01-03 14:30:00 110052.SH 0          180.0     129.580        100.0   
                              1           10.0     129.450         10.0   
                              2           10.0     129.450         10.0   
                              3           10.0     129.450         10.0   
                              4           40.0     129.349         60.0   

                                   ask_price5  ask_volume5  bid_price5  \
dt                  code      seq                                        
2025-01-03 14:30:00 110052.SH 0       129.887         20.0     129.560   
                              1       129.658         30.0     129.424   
                              2       129.658         30.0     129.424   
                              3       129.613        110.0     129.424   
                              4       129.605         10.0     129.347   

                                   bid_volume5  iopv  trading_phase_code  
dt                  code      seq                                         
2025-01-03 14:30:00 110052.SH 0           50.0   0.0                   T  
                              1           30.0   0.0                   T  
                              2           30.0   0.0                   T  
                              3           30.0   0.0                   T  
                              4          100.0   0.0                   T  

[5 rows x 34 columns]

## 4) Label Data (print head)

In [ ]:
label_root = Path(paths_cfg['label_data_root'])
label_files = list(label_root.rglob(f"*{DATE.strftime('%Y%m%d')}*.parquet")) if label_root.exists() else []
print('label files:', len(label_files))
if label_files:
    df = pd.read_parquet(label_files[0])
    display(df.head())
df

label files: 1


,code,trade_time,y
0,110052.SH,2025-01-03 14:45:00,-0.003140
1,110055.SH,2025-01-03 14:45:00,-0.018813
2,110059.SH,2025-01-03 14:45:00,-0.000360
3,110060.SH,2025-01-03 14:45:00,-0.016699
4,110062.SH,2025-01-03 14:45:00,0.003054


## 5) Factor Data (print head)

In [19]:
factor_root = Path(paths_cfg['factor_data_root'])
factor_files = list(factor_root.rglob(f"*{DATE.strftime('%Y%m%d')}*.parquet")) if factor_root.exists() else []
print('factor files:', len(factor_files))
if factor_files:
    df = pd.read_parquet(factor_files[0])
    display(df.head())


factor files: 1


aacb_3_  volen_60_10_3_
dt                  code                               
2025-01-03 14:30:00 110052.SH  0.001083        1.217638
                    110055.SH  0.001096        1.146982
                    110059.SH  0.000163        0.550660
                    110060.SH  0.000671        0.922992
                    110062.SH  0.000708        2.362671

## 6) Results (list latest output folders)

In [20]:
results_root = Path(paths_cfg['results_root'])
sorted(results_root.iterdir())[:5] if results_root.exists() else f'Missing: {results_root}'


[WindowsPath('D:/cbond_on/results/2025-01-01_2026-01-28'),
 WindowsPath('D:/cbond_on/results/models')]